In [ ]:
# SETUP - Self-contained, works anywhere

import pandas as pd
import numpy as np
import sqlite3
import os
import json
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Auto-detect environment
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    WORKSPACE = '/workspaces/quantum-ai-trader_v1.1'
    ENV = 'Codespace'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    WORKSPACE = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
    ENV = 'Shadow PC'
else:
    WORKSPACE = str(Path.cwd().parent)
    ENV = 'JupyterLab'

DATA_DIR = os.path.join(WORKSPACE, 'data')
DB_PATH = os.path.join(DATA_DIR, 'trading_system.db')

print(f"✅ Environment: {ENV}")
print(f"📁 Database: {DB_PATH}")
print(f"🕐 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n🎯 Mission: Backtest 3 scanners on 3,448 real events")
print(f"🎯 Goal: Find scanner with >60% win rate")
print(f"🎯 Prize: Build live scanner with real capital\n")

In [ ]:
# CONNECT TO DATABASE

conn = sqlite3.connect(DB_PATH)

# Sanity check
check_query = """
    SELECT COUNT(DISTINCT ticker) as tickers,
           MIN(date) as earliest,
           MAX(date) as latest,
           COUNT(*) as total_bars
    FROM ohlcv_daily
"""
stats = pd.read_sql_query(check_query, conn)
print(f"📊 Database: {stats['tickers'].iloc[0]} tickers, {stats['total_bars'].iloc[0]:,} bars")
print(f"📅 Range: {stats['earliest'].iloc[0]} to {stats['latest'].iloc[0]}")

if stats['tickers'].iloc[0] < 300:
    print("\n⚠️  WARNING: Less than 300 tickers - run DAY1_DATA_COLLECTION.ipynb first")
    raise SystemExit("Insufficient data for backtest")

## STEP 1: Load All Big Events (3,448 moves)

These are our **test cases**. Each scanner will be judged on how well it catches these.

In [ ]:
# LOAD BIG EVENTS - Our ground truth dataset

events_query = """
    WITH daily_returns AS (
        SELECT 
            ticker,
            date,
            open,
            close,
            volume,
            LAG(close, 1) OVER (PARTITION BY ticker ORDER BY date) as prev_close,
            LAG(open, 1) OVER (PARTITION BY ticker ORDER BY date) as prev_open,
            AVG(volume) OVER (
                PARTITION BY ticker 
                ORDER BY date 
                ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
            ) as avg_volume_20d
        FROM ohlcv_daily
        WHERE date >= date('now', '-12 months')
    )
    SELECT 
        ticker,
        date,
        open,
        close,
        prev_close,
        prev_open,
        volume,
        avg_volume_20d,
        ROUND(((close - prev_close) / prev_close * 100), 2) as pct_change,
        ROUND((volume / avg_volume_20d), 2) as volume_ratio
    FROM daily_returns
    WHERE prev_close IS NOT NULL
      AND avg_volume_20d > 0
      AND ABS((close - prev_close) / prev_close) >= 0.10
    ORDER BY date, ticker
"""

big_events = pd.read_sql_query(events_query, conn)
big_events['date'] = pd.to_datetime(big_events['date'])

print(f"📊 Loaded {len(big_events)} big events (10%+ moves)")
print(f"📅 Date range: {big_events['date'].min()} to {big_events['date'].max()}")
print(f"\n✅ These are our test cases - scanners will be judged on catching these\n")

## STEP 2: Scanner 1 - VOLUME BREAKOUT

**Strategy:** Buy when volume >20x average AND price >5%  
**Why it works:** Insider buying shows up as massive volume  
**Examples:** SPRO (1,963x vol → +244%), XBIO (1,595x vol → +141%)

In [ ]:
# SCANNER 1: VOLUME BREAKOUT

def scanner_volume_breakout(ticker, date, volume, avg_volume, price_change, min_vol_ratio=20, min_price_change=5):
    """
    Triggers when:
    - Volume is >20x average (default)
    - Price is up >5% (default)
    
    Returns True if signal triggered, False otherwise.
    """
    if avg_volume == 0:
        return False
    
    vol_ratio = volume / avg_volume
    
    return (vol_ratio >= min_vol_ratio) and (price_change >= min_price_change)

# Test Scanner 1 on all events
print("🔍 Testing Scanner 1: VOLUME BREAKOUT\n")
print("Trigger: Volume >20x average + Price >5%\n")

scanner1_signals = []

for idx, event in big_events.iterrows():
    triggered = scanner_volume_breakout(
        event['ticker'],
        event['date'],
        event['volume'],
        event['avg_volume_20d'],
        event['pct_change']
    )
    
    if triggered:
        scanner1_signals.append({
            'ticker': event['ticker'],
            'date': event['date'],
            'pct_change': event['pct_change'],
            'volume_ratio': event['volume_ratio'],
            'entry_price': event['close'],  # Would buy at close of signal day
            'signal_type': 'volume_breakout'
        })

scanner1_df = pd.DataFrame(scanner1_signals)

print(f"📊 Results:")
print(f"   Total events: {len(big_events)}")
print(f"   Scanner 1 triggered: {len(scanner1_df)} times ({len(scanner1_df)/len(big_events)*100:.1f}%)")
print(f"   Avg move when triggered: {scanner1_df['pct_change'].mean():.1f}%")
print(f"\n🔥 Top 10 Scanner 1 signals:")
print(scanner1_df.nlargest(10, 'pct_change')[['ticker', 'date', 'pct_change', 'volume_ratio']])

## STEP 3: Scanner 2 - MOMENTUM CONTINUATION

**Strategy:** Buy stocks that moved 10%+ in last 30 days when they spike 5%+ again  
**Why it works:** Hot stocks stay hot (volatility clustering)  
**Examples:** BYND moved +127% then +146% next day

In [ ]:
# SCANNER 2: MOMENTUM CONTINUATION

def scanner_momentum_continuation(ticker, date, conn, lookback_days=30, prior_move_threshold=10, current_move_threshold=5):
    """
    Triggers when:
    - Ticker had 10%+ move in last 30 days
    - Ticker is now moving 5%+ again
    
    Returns True if signal triggered.
    """
    # Check if ticker had big move in last 30 days
    prior_query = f"""
        WITH daily_returns AS (
            SELECT 
                date,
                close,
                LAG(close, 1) OVER (ORDER BY date) as prev_close
            FROM ohlcv_daily
            WHERE ticker = '{ticker}'
              AND date BETWEEN date('{date}', '-{lookback_days} days') 
                           AND date('{date}', '-1 day')
        )
        SELECT MAX(ABS((close - prev_close) / prev_close * 100)) as max_move
        FROM daily_returns
        WHERE prev_close IS NOT NULL
    """
    
    result = pd.read_sql_query(prior_query, conn)
    
    if result.empty or result['max_move'].iloc[0] is None:
        return False
    
    max_prior_move = result['max_move'].iloc[0]
    
    return max_prior_move >= prior_move_threshold

# Test Scanner 2 on subset (full test takes longer)
print("🔍 Testing Scanner 2: MOMENTUM CONTINUATION\n")
print("Trigger: Prior 10%+ move in 30 days + Current 5%+ move\n")
print("⚠️  Note: Testing on subset (first 500 events) for speed...\n")

scanner2_signals = []

# Test on gainers only (pct_change > 5%)
test_events = big_events[big_events['pct_change'] >= 5].head(500)

for idx, event in test_events.iterrows():
    triggered = scanner_momentum_continuation(
        event['ticker'],
        event['date'].strftime('%Y-%m-%d'),
        conn
    )
    
    if triggered:
        scanner2_signals.append({
            'ticker': event['ticker'],
            'date': event['date'],
            'pct_change': event['pct_change'],
            'volume_ratio': event['volume_ratio'],
            'entry_price': event['close'],
            'signal_type': 'momentum_continuation'
        })

scanner2_df = pd.DataFrame(scanner2_signals)

print(f"📊 Results (subset):")
print(f"   Events tested: {len(test_events)}")
print(f"   Scanner 2 triggered: {len(scanner2_df)} times ({len(scanner2_df)/len(test_events)*100:.1f}%)")
if len(scanner2_df) > 0:
    print(f"   Avg move when triggered: {scanner2_df['pct_change'].mean():.1f}%")
    print(f"\n🔥 Top 10 Scanner 2 signals:")
    print(scanner2_df.nlargest(10, 'pct_change')[['ticker', 'date', 'pct_change', 'volume_ratio']])
else:
    print(f"\n⚠️  No signals triggered in test subset")

## STEP 4: Scanner 3 - PRE-EVENT VOLUME

**Strategy:** Buy when volume spikes 2x+ for 2+ days BUT price hasn't moved yet  
**Why it works:** 47% of big moves had volume spikes BEFORE price moved  
**Examples:** Catches accumulation before explosion

In [ ]:
# SCANNER 3: PRE-EVENT VOLUME

def scanner_pre_event_volume(ticker, date, conn, lookback_days=3, vol_threshold=2.0, max_price_move=3.0):
    """
    Triggers when:
    - Volume was 2x+ average for 2+ of last 3 days
    - But price hasn't moved much (<3%)
    
    Returns True if accumulation pattern detected.
    """
    query = f"""
        WITH baseline AS (
            SELECT AVG(volume) as avg_vol
            FROM ohlcv_daily
            WHERE ticker = '{ticker}'
              AND date BETWEEN date('{date}', '-30 days') 
                           AND date('{date}', '-{lookback_days+1} days')
        ),
        recent AS (
            SELECT 
                date,
                close,
                volume,
                LAG(close, 1) OVER (ORDER BY date) as prev_close,
                (SELECT avg_vol FROM baseline) as baseline_volume
            FROM ohlcv_daily
            WHERE ticker = '{ticker}'
              AND date BETWEEN date('{date}', '-{lookback_days} days') 
                           AND date('{date}', '-1 day')
        )
        SELECT 
            COUNT(CASE WHEN volume > baseline_volume * {vol_threshold} THEN 1 END) as high_vol_days,
            MAX(ABS((close - prev_close) / prev_close * 100)) as max_price_move
        FROM recent
        WHERE baseline_volume > 0
    """
    
    result = pd.read_sql_query(query, conn)
    
    if result.empty:
        return False
    
    high_vol_days = result['high_vol_days'].iloc[0] or 0
    max_move = result['max_price_move'].iloc[0] or 0
    
    # Triggered if 2+ high volume days but price stayed flat
    return (high_vol_days >= 2) and (max_move < max_price_move)

# Test Scanner 3 on subset
print("🔍 Testing Scanner 3: PRE-EVENT VOLUME\n")
print("Trigger: 2x+ volume for 2+ days, but price flat (<3%)\n")
print("⚠️  Note: Testing on subset (first 500 events) for speed...\n")

scanner3_signals = []

test_events = big_events.head(500)

for idx, event in test_events.iterrows():
    triggered = scanner_pre_event_volume(
        event['ticker'],
        event['date'].strftime('%Y-%m-%d'),
        conn
    )
    
    if triggered:
        scanner3_signals.append({
            'ticker': event['ticker'],
            'date': event['date'],
            'pct_change': event['pct_change'],
            'volume_ratio': event['volume_ratio'],
            'entry_price': event['prev_close'],  # Would have entered BEFORE the move
            'signal_type': 'pre_event_volume'
        })

scanner3_df = pd.DataFrame(scanner3_signals)

print(f"📊 Results (subset):")
print(f"   Events tested: {len(test_events)}")
print(f"   Scanner 3 triggered: {len(scanner3_df)} times ({len(scanner3_df)/len(test_events)*100:.1f}%)")
if len(scanner3_df) > 0:
    print(f"   Avg move when triggered: {scanner3_df['pct_change'].mean():.1f}%")
    print(f"\n🔥 Top 10 Scanner 3 signals:")
    print(scanner3_df.nlargest(10, 'pct_change')[['ticker', 'date', 'pct_change', 'volume_ratio']])
else:
    print(f"\n⚠️  No signals triggered in test subset")

## STEP 5: Scanner Comparison

**Which scanner caught the most winners?**

In [ ]:
# SCANNER COMPARISON

print("="*80)
print("📊 SCANNER BATTLE RESULTS")
print("="*80)

scanners = [
    ('Scanner 1: Volume Breakout', scanner1_df),
    ('Scanner 2: Momentum Continuation', scanner2_df),
    ('Scanner 3: Pre-Event Volume', scanner3_df)
]

for name, df in scanners:
    print(f"\n{name}:")
    print(f"  Signals: {len(df)}")
    if len(df) > 0:
        winners = len(df[df['pct_change'] > 0])
        losers = len(df[df['pct_change'] < 0])
        win_rate = winners / len(df) * 100
        avg_gain = df['pct_change'].mean()
        
        print(f"  Winners: {winners} | Losers: {losers}")
        print(f"  Win Rate: {win_rate:.1f}%")
        print(f"  Avg Gain: {avg_gain:.1f}%")
        
        if win_rate >= 60:
            print(f"  ✅ PASSED: Win rate >60%")
        else:
            print(f"  ❌ FAILED: Win rate <60%")
    else:
        print(f"  ⚠️  No signals generated")

print("\n" + "="*80)
print("\n💡 NOTE: Scanners 2 & 3 tested on subset (500 events).")
print("   Run full backtest for complete results.\n")

## NEXT STEPS

**If you're stopping here tonight:**
1. Review scanner results above
2. Tomorrow: Run full backtest (all 3,448 events)
3. Pick winner based on win rate
4. Build live scanner

**To continue now:**
1. Expand Scanner 2 & 3 to test all events (remove `.head(500)`)
2. Add exit logic (next day close, 20% target, 10% stop)
3. Calculate Sharpe ratio, max drawdown
4. Export winning scanner to production

---

**The data is speaking. Are you listening?**

In [ ]:
# EXPLORATION: Repeat Winners - Tickers that moved big multiple times

print("🔍 EXPLORATION 1: Repeat Winners\n")
print("Tickers that had multiple 10%+ moves (volatility clustering):\n")

repeat_movers = big_events['ticker'].value_counts()
repeat_df = repeat_movers[repeat_movers >= 3].reset_index()
repeat_df.columns = ['ticker', 'num_big_moves']

print(f"Found {len(repeat_df)} tickers with 3+ big moves:\n")
print(repeat_df.head(20))

print("\n💡 INSIGHT: These are 'hot' tickers - trade them when they spike again")
print("   Potential Scanner 4: 'Hot Ticker Alert' when these move >5%\n")

In [ ]:
# EXPLORATION: Extreme Volume - When volume goes REALLY crazy

print("🔍 EXPLORATION 2: Extreme Volume Events\n")
print("Events with volume >100x average:\n")

extreme_vol = big_events[big_events['volume_ratio'] > 100].sort_values('volume_ratio', ascending=False)

print(f"Found {len(extreme_vol)} events with >100x volume:\n")
print(extreme_vol[['ticker', 'date', 'pct_change', 'volume_ratio']].head(20))

print("\n💡 INSIGHT: Extreme volume (>100x) often = news/catalyst")
print(f"   Average move: {extreme_vol['pct_change'].mean():.1f}%")
print("   These are RARE but HUGE opportunities\n")

In [ ]:
# EXPLORATION: Gap Patterns - Overnight gaps

print("🔍 EXPLORATION 3: Gap Analysis\n")
print("Looking for overnight gaps (open vs prev close):\n")

# Calculate gaps
big_events['gap_pct'] = ((big_events['open'] - big_events['prev_close']) / big_events['prev_close'] * 100)
big_gaps = big_events[big_events['gap_pct'].abs() >= 5]

print(f"Found {len(big_gaps)} events with >5% overnight gap:\n")
print(big_gaps[['ticker', 'date', 'gap_pct', 'pct_change']].head(20))

gap_up = big_gaps[big_gaps['gap_pct'] > 0]
gap_down = big_gaps[big_gaps['gap_pct'] < 0]

print(f"\n📊 Gap Statistics:")
print(f"   Gap ups: {len(gap_up)} (avg move: {gap_up['pct_change'].mean():.1f}%)")
print(f"   Gap downs: {len(gap_down)} (avg move: {gap_down['pct_change'].mean():.1f}%)")

print("\n💡 INSIGHT: Can we trade gap continuation or reversal?\n")

In [ ]:
# EXPLORATION SUMMARY - All findings visible in notebook output

print("="*80)
print("📊 DISCOVERY SUMMARY - DAY 3 SCANNER BACKTEST")
print("="*80)
print(f"Session: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

findings_count = 0

# Finding 1: Repeat Winners
if len(repeat_df) > 0:
    findings_count += 1
    print(f"\n🔥 FINDING {findings_count}: REPEAT WINNERS")
    print("-" * 80)
    print(f"Count: {len(repeat_df)} tickers with 3+ big moves")
    print(f"Top 10:")
    for idx, row in repeat_df.head(10).iterrows():
        print(f"  {row['ticker']}: {row['num_big_moves']} big moves")
    print(f"\n💡 Potential Scanner: 'Hot Ticker Alert' - watch these for 5%+ moves")

# Finding 2: Extreme Volume
if len(extreme_vol) > 0:
    findings_count += 1
    print(f"\n🔥 FINDING {findings_count}: EXTREME VOLUME EVENTS")
    print("-" * 80)
    print(f"Count: {len(extreme_vol)} events with >100x volume")
    print(f"Average move: {extreme_vol['pct_change'].mean():.1f}%")
    print(f"Top 5 extreme volume:")
    for idx, row in extreme_vol.head(5).iterrows():
        print(f"  {row['ticker']} on {row['date'].strftime('%Y-%m-%d')}: {row['volume_ratio']:.0f}x vol → {row['pct_change']:.1f}%")
    print(f"\n💡 Potential Scanner: 'News Catalyst Scanner' - extreme volume = breaking news")

# Finding 3: Gap Patterns
if len(big_gaps) > 0:
    findings_count += 1
    print(f"\n🔥 FINDING {findings_count}: OVERNIGHT GAP PATTERNS")
    print("-" * 80)
    print(f"Total gaps >5%: {len(big_gaps)}")
    print(f"Gap ups: {len(gap_up)} (avg day move: {gap_up['pct_change'].mean():.1f}%)")
    print(f"Gap downs: {len(gap_down)} (avg day move: {gap_down['pct_change'].mean():.1f}%)")
    print(f"\n💡 Potential Scanner: 'Gap Continuation/Reversal Scanner'")
print("\n" + "="*80)
print("💡 NEXT ACTIONS:")
print("="*80)
print(f"Total discoveries: {findings_count}")
print("\n1. Review scanner battle results (Scanner 1/2/3 above)")
print("2. Review exploration findings (repeat winners, extreme volume, gaps)")
print("3. Pick best scanner OR build hybrid scanner")
print("4. Test winning approach with exit logic")
print("5. Commit notebook to GitHub (all results saved in outputs)")
print("\n**The gold is in the anomalies. Keep exploring.**")
print("**All results saved in THIS notebook - commit to GitHub when done.**\n")
